# GradeScope — 06. Eksport modeli do ONNX

**Cel:** eksport Pipeline'ów (StandardScaler + model) do plików .onnx oraz weryfikacja poprawności

In [1]:
import pickle
import json
import numpy as np
import pandas as pd
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

# Wczytanie danych i modeli
with open("../data/splits.pkl", "rb") as f:
    data = pickle.load(f)

with open("../data/selected_features.pkl", "rb") as f:
    selected_features = pickle.load(f)

with open("../data/trained_pipelines.pkl", "rb") as f:
    pipelines = pickle.load(f)

X_train = data["X_train"][selected_features]
X_test  = data["X_test"][selected_features]
y_test  = data["y_test"]
n_features = len(selected_features)

print(f"Modele: {list(pipelines.keys())}")
print(f"Cechy: {n_features} — {selected_features}")

Modele: ['random_forest', 'svm', 'gradient_boosting']
Cechy: 12 — ['Attendance', 'Hours_Studied', 'Previous_Scores', 'Tutoring_Sessions', 'Sleep_Hours', 'Parental_Involvement', 'Access_to_Resources', 'Physical_Activity', 'Peer_Influence', 'Family_Income', 'Parental_Education_Level', 'Distance_from_Home']


## 1. Eksport Pipeline'ów do ONNX

In [2]:
initial_type = [("float_input", FloatTensorType([None, n_features]))]

# zipmap=False — probability output as float32 tensor instead of sequence<map>
# Required for onnxruntime-web compatibility
for name, pipeline in pipelines.items():
    onnx_model = convert_sklearn(pipeline, initial_types=initial_type, options={'zipmap': False})
    path = f"../models/{name}.onnx"
    with open(path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    size_kb = len(onnx_model.SerializeToString()) / 1024
    print(f"✓ {name}.onnx — {size_kb:.1f} KB")

✓ random_forest.onnx — 5477.1 KB
✓ svm.onnx — 103.0 KB
✓ gradient_boosting.onnx — 54.6 KB


## 2. Weryfikacja — ONNX vs sklearn

In [3]:
# Porównanie predykcji sklearn vs ONNX na zbiorze testowym
X_test_f32 = X_test.values.astype(np.float32)

for name in pipelines:
    sess = rt.InferenceSession(f"../models/{name}.onnx")
    input_name = sess.get_inputs()[0].name

    pred_onnx    = sess.run(None, {input_name: X_test_f32})[0]
    pred_sklearn = pipelines[name].predict(X_test)

    zgodnosc = np.mean(pred_onnx == pred_sklearn) * 100
    print(f"{name}: zgodność ONNX vs sklearn = {zgodnosc:.2f}%")

random_forest: zgodność ONNX vs sklearn = 99.55%
svm: zgodność ONNX vs sklearn = 100.00%
gradient_boosting: zgodność ONNX vs sklearn = 100.00%


## 3. Zapis metadanych dla frontendu

In [4]:
# Zakresy cech z danych treningowych — potrzebne dla suwaków frontendu
df_train = data["X_train"][selected_features]

feature_meta = {}
for col in selected_features:
    feature_meta[col] = {
        "min":  int(df_train[col].min()),
        "max":  int(df_train[col].max()),
        "mean": round(float(df_train[col].mean()), 2),
    }

metadata = {
    "features": selected_features,
    "feature_meta": feature_meta,
    "models": list(pipelines.keys()),
    "pass_threshold": 67,
}

with open("../data/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Zapisano: model_metadata.json")
print(json.dumps(feature_meta, indent=2))

Zapisano: model_metadata.json
{
  "Attendance": {
    "min": 60,
    "max": 100,
    "mean": 79.91
  },
  "Hours_Studied": {
    "min": 1,
    "max": 44,
    "mean": 19.93
  },
  "Previous_Scores": {
    "min": 50,
    "max": 100,
    "mean": 75.17
  },
  "Tutoring_Sessions": {
    "min": 0,
    "max": 8,
    "mean": 1.5
  },
  "Sleep_Hours": {
    "min": 4,
    "max": 10,
    "mean": 7.04
  },
  "Parental_Involvement": {
    "min": 0,
    "max": 2,
    "mean": 1.22
  },
  "Access_to_Resources": {
    "min": 0,
    "max": 2,
    "mean": 1.2
  },
  "Physical_Activity": {
    "min": 0,
    "max": 6,
    "mean": 2.97
  },
  "Peer_Influence": {
    "min": 0,
    "max": 2,
    "mean": 1.19
  },
  "Family_Income": {
    "min": 0,
    "max": 2,
    "mean": 1.21
  },
  "Parental_Education_Level": {
    "min": 0,
    "max": 2,
    "mean": 0.89
  },
  "Distance_from_Home": {
    "min": 0,
    "max": 2,
    "mean": 1.5
  }
}
